In [1]:
import pandas as pd

df_clean = pd.read_parquet('../data/curated/part1_taxi_curated.parquet')
df_clean.sample(5, random_state=42)


,PULocationID,DOLocationID,PU_borough,PU_zone,DO_borough,DO_zone,tpep_pickup_datetime,tpep_dropoff_datetime,pickup_date,pickup_hour,...,trip_distance,fare_amount,tip_amount,total_amount,payment_type,trip_duration_min,speed_mph,fare_per_mile,tip_rate,distance_bucket
2409739,237,48,Manhattan,Upper East Side South,Manhattan,Clinton East,2023-01-26 22:42:40,2023-01-26 22:55:51,2023-01-26,22,...,2.10,14.2,3.84,23.04,1,13.183333,9.557522,6.761905,0.166667,"[1,3)"
1442295,170,161,Manhattan,Murray Hill,Manhattan,Midtown Center,2023-01-17 08:33:17,2023-01-17 08:42:16,2023-01-17,8,...,0.83,9.3,2.66,15.96,1,8.983333,5.543599,11.204819,0.166667,"[0,1)"
2803684,43,161,Manhattan,Central Park,Manhattan,Midtown Center,2023-01-31 09:26:42,2023-01-31 09:36:13,2023-01-31,9,...,1.87,11.4,0.00,15.40,2,9.516667,11.789842,6.096257,0.000000,"[1,3)"
1988338,148,114,Manhattan,Lower East Side,Manhattan,Greenwich Village South,2023-01-22 14:35:04,2023-01-22 14:48:41,2023-01-22,14,...,1.26,12.8,3.36,20.16,1,13.616667,5.552020,10.158730,0.166667,"[1,3)"
1052786,158,48,Manhattan,Meatpacking/West Village West,Manhattan,Clinton East,2023-01-13 00:20:57,2023-01-13 00:30:25,2023-01-13,0,...,2.69,13.5,1.00,19.50,1,9.466667,17.049296,5.018587,0.051282,"[1,3)"


## A. Required groupby metrics.

### Non-trivial Statistics

**1. Daily trip counts grouped by PU_borough and pickup_date.**
   
   Grouping by `PU_borough` and `pickup_date` to get the number of trips per borough per day.


In [2]:
# 1. Daily trip counts grouped by PU_borough and pickup_date.
daily_trip_counts = (
    df_clean.groupby(['PU_borough', 'pickup_date'])
    .size()
    .reset_index(name='trip_count')
)

print(f"\n--- Daily Trip Counts --- {daily_trip_counts.shape}")
daily_trip_counts



--- Daily Trip Counts --- (217, 3)


,PU_borough,pickup_date,trip_count
0,Bronx,2023-01-01,91
1,Bronx,2023-01-02,61
2,Bronx,2023-01-03,122
3,Bronx,2023-01-04,165
4,Bronx,2023-01-05,140
...,...,...,...
212,Unknown,2023-01-27,1329
213,Unknown,2023-01-28,1204
214,Unknown,2023-01-29,1282
215,Unknown,2023-01-30,1089


**2. Mean and median of fare_amount and trip_distance grouped by PU_borough and weekday**
   
   Grouping by `PU_borough` and `weekday` to get the mean and median of fare_amount and trip_distance for each borough and weekday.


In [3]:
# 2. Mean and median of fare_amount and trip_distance grouped by PU_borough and weekday

fare_dist_stats = (
    df_clean.groupby(["PU_borough", "weekday"])[["fare_amount", "trip_distance"]]
    .agg(["mean", "median"])
    .reset_index()
    .round(4)
)

print(f"--- Fare and Distance Stats --- {fare_dist_stats.shape}")
fare_dist_stats.head()

--- Fare and Distance Stats --- (49, 6)


PU_borough weekday fare_amount        trip_distance       
                            mean median          mean median
0      Bronx     Fri     31.8469   26.2        7.2717   4.69
1      Bronx     Mon     31.4150   29.2        7.3752   5.70
2      Bronx     Sat     27.6058   22.5        6.6523   4.25
3      Bronx     Sun     26.8621   22.5        5.8563   4.40
4      Bronx     Thu     29.7251   25.4        6.8987   4.70

**3. For each pickup_hour, the proportion of trips by payment_type**

The `normalize=True` argument ensures that the proportions are calculated as a percentage of the total number of trips for that pickup_hour.

In [4]:
# 3. For each pickup_hour, the proportion of trips by payment_type
payment_type_props = (
    df_clean.groupby("pickup_hour")["payment_type"]
    .value_counts(normalize=True)
    .reset_index(name="proportion")
    .round(4)
)

print(f"\n--- Payment Type Proportions --- {payment_type_props.shape}")
payment_type_props.head()


--- Payment Type Proportions --- (96, 3)


,pickup_hour,payment_type,proportion
0,0,1,0.8300
1,0,2,0.1575
2,0,4,0.0083
3,0,3,0.0043
4,1,1,0.8419


**4. Mean tip_rate grouped by distance_bucket**

In [5]:
# 4. Mean tip_rate grouped by distance_bucket
mean_tip_rate_by_distance = (
    df_clean.groupby("distance_bucket")["tip_rate"]
    .mean()
    .reset_index(name="mean_tip_rate")
    .round(4)
)

print(f"\n--- Mean Tip Rate by Distance Bucket --- {mean_tip_rate_by_distance.shape}")
mean_tip_rate_by_distance


--- Mean Tip Rate by Distance Bucket --- (5, 2)


,distance_bucket,mean_tip_rate
0,"[0,1)",0.1191
1,"[1,3)",0.1237
2,"[3,5)",0.1205
3,"[5,10)",0.1128
4,"[10,+)",0.1110


## B. Required window-style metrics.

In [6]:
daily_trip_counts = daily_trip_counts.sort_values(
    by=["PU_borough", "pickup_date"]
).reset_index(drop=True)

daily_trip_counts

,PU_borough,pickup_date,trip_count
0,Bronx,2023-01-01,91
1,Bronx,2023-01-02,61
2,Bronx,2023-01-03,122
3,Bronx,2023-01-04,165
4,Bronx,2023-01-05,140
...,...,...,...
212,Unknown,2023-01-27,1329
213,Unknown,2023-01-28,1204
214,Unknown,2023-01-29,1282
215,Unknown,2023-01-30,1089


**1. For each PU_borough, a 7-day rolling mean of daily trip counts ordered by pickup_date**

The `rolling(window=7, min_periods=1)` argument specifies a window of 7 days and a minimum of 1 day required to calculate the rolling mean. This allows for the calculation of the rolling mean even when there are fewer than 7 days of data available such as at the first few days. 

In [7]:
# 1. For each PU_borough, a 7-day rolling mean of daily trip counts ordered by pickup_date
daily_trip_counts["rolling_7d_mean"] = (
    daily_trip_counts
    .sort_values(by='pickup_date', ascending=True)
    .groupby("PU_borough")["trip_count"]
    .transform(lambda x: x.rolling(window=7, min_periods=1)
    .mean()
    .round(4)
))

daily_trip_counts

,PU_borough,pickup_date,trip_count,rolling_7d_mean
0,Bronx,2023-01-01,91,91.0000
1,Bronx,2023-01-02,61,76.0000
2,Bronx,2023-01-03,122,91.3333
3,Bronx,2023-01-04,165,109.7500
4,Bronx,2023-01-05,140,115.8000
...,...,...,...,...
212,Unknown,2023-01-27,1329,1165.4286
213,Unknown,2023-01-28,1204,1171.1429
214,Unknown,2023-01-29,1282,1175.4286
215,Unknown,2023-01-30,1089,1180.5714


**2. For each PU_borough, the day-over-day difference and day-over-day percent change**

The `diff()` and `pct_change()` methods are used to calculate the day-over-day difference and day-over-day percent change respectively.


In [8]:
# 2. For each PU_borough, the day-over-day difference and day-over-day percent change
daily_trip_counts["dod_diff"] = (daily_trip_counts
    .groupby("PU_borough")["trip_count"]
    .diff()
)


daily_trip_counts["dod_pct_change"] = (daily_trip_counts
    .groupby("PU_borough")["trip_count"]
    .pct_change()
)

daily_trip_counts

,PU_borough,pickup_date,trip_count,rolling_7d_mean,dod_diff,dod_pct_change
0,Bronx,2023-01-01,91,91.0000,NaN,NaN
1,Bronx,2023-01-02,61,76.0000,-30.0,-0.329670
2,Bronx,2023-01-03,122,91.3333,61.0,1.000000
3,Bronx,2023-01-04,165,109.7500,43.0,0.352459
4,Bronx,2023-01-05,140,115.8000,-25.0,-0.151515
...,...,...,...,...,...,...
212,Unknown,2023-01-27,1329,1165.4286,54.0,0.042353
213,Unknown,2023-01-28,1204,1171.1429,-125.0,-0.094056
214,Unknown,2023-01-29,1282,1175.4286,78.0,0.064784
215,Unknown,2023-01-30,1089,1180.5714,-193.0,-0.150546


## C. Required EDA questions (answer all three with evidence).

1. Which borough has the strongest weekday effect? Quantify it using:

$$
S_{borough} = \frac{\max({\mu_{\text{weekday,borough}}})-{\min({\mu_{\text{weekday,borough}}})}}{\mu_{\text{borough}}}
$$

where $S_{\text{borough}}$ is the measure of weekday effect , µweekday, borough is mean daily trips for that weekday in the borough and µweekday, borough is mean daily trips for the borough.

In [9]:
# mean daily trips per (borough, weekday)
# first add weekday column to daily_trip_counts
daily_trip_counts['weekday'] = pd.to_datetime(daily_trip_counts['pickup_date']).dt.day_name()

mu_weekday = (
    daily_trip_counts
    .groupby(['PU_borough', 'weekday'])['trip_count']
    .mean()
    .reset_index(name='mu_weekday')
)

# overall mean daily trips per borough
mu_borough = (
    daily_trip_counts
    .groupby('PU_borough')['trip_count']
    .mean()
    .reset_index(name='mu_borough')
)

# S_borough = (max weekday mean - min weekday mean) / overall mean
s_df = (
    mu_weekday
    .groupby('PU_borough')['mu_weekday']
    .agg(max_mu='max', min_mu='min')
    .reset_index()
    .merge(mu_borough, on='PU_borough') 
)
s_df['S_borough'] = (s_df['max_mu'] - s_df['min_mu']) / s_df['mu_borough'] #calculate weekday effect score for each borough

print('---Weekday Effect S_borough per borough ---')
print(s_df.sort_values('S_borough', ascending=False).to_string(index=False))

---Weekday Effect S_borough per borough ---
   PU_borough   max_mu       min_mu   mu_borough  S_borough
          EWR     4.20     1.500000     3.080000   0.876623
        Bronx   121.50    62.800000    92.258065   0.636259
       Queens  9611.20  5398.833333  7662.485714   0.549739
     Brooklyn   538.75   326.400000   427.419355   0.496819
    Manhattan 93265.00 64964.200000 77631.424242   0.364553
Staten Island     9.50     7.200000     8.290323   0.277432
      Unknown  1329.00  1073.400000  1208.129032   0.211567


### C2. Identify two specific dates with anomalous total trip volume using a z-score on daily totals.
### Show the z-scores and the two selected dates.


In [10]:
# daily total trips across  boroughs
daily_total = (
    daily_trip_counts
    .groupby('pickup_date')['trip_count']
    .sum()
    .reset_index(name='total_trips')
)

daily_total['pickup_date'] = pd.to_datetime(daily_total['pickup_date'])

 
daily_total_ = daily_total[
    (daily_total['pickup_date'].dt.year == 2023) &
    (daily_total['pickup_date'].dt.month == 1)
].copy()

# compute z-score
mu  = daily_total_['total_trips'].mean()
sig = daily_total_['total_trips'].std()
daily_total_['z_score'] = ((daily_total_['total_trips'] - mu) / sig).round(4)

print(daily_total_.sort_values('z_score').to_string(index=False))

# two most anomalous dates
daily_total_['abs_z'] = daily_total_['z_score'].abs()
anomalies = daily_total_.nlargest(2, 'abs_z')
print('\n--- Two most anomalous dates ---')
print(anomalies[['pickup_date', 'total_trips', 'z_score']].to_string(index=False))

pickup_date  total_trips  z_score
 2023-01-02        61878  -2.5501
 2023-01-01        70004  -1.8849
 2023-01-16        75501  -1.4349
 2023-01-30        78646  -1.1774
 2023-01-09        80076  -1.0603
 2023-01-08        80167  -1.0529
 2023-01-03        80561  -1.0206
 2023-01-29        82848  -0.8334
 2023-01-23        84242  -0.7193
 2023-01-22        84243  -0.7192
 2023-01-04        89522  -0.2870
 2023-01-15        91120  -0.1562
 2023-01-10        94322   0.1059
 2023-01-31        94475   0.1184
 2023-01-17        95092   0.1689
 2023-01-05        95188   0.1768
 2023-01-06        96629   0.2948
 2023-01-24        97906   0.3993
 2023-01-07        99189   0.5043
 2023-01-18       100151   0.5831
 2023-01-11       100173   0.5849
 2023-01-25       102145   0.7463
 2023-01-20       103197   0.8324
 2023-01-13       104124   0.9083
 2023-01-12       104470   0.9367
 2023-01-28       105147   0.9921
 2023-01-27       105199   0.9963
 2023-01-21       105665   1.0345
 2023-01-14   

### Determine whether tip_rate is more strongly associated with pickup_hour or distance_bucket.
### Use a quantitative comparison and report your conclusion.

In [11]:

# higher = stronger association

clean_tips = df_clean[df_clean['tip_rate'].notna()].copy() # keep rows with no null tip_rate for this analysis

def eta_squared(groups, values):
    grand_mean = values.mean()
    ss_between = sum(
        len(values[groups == g]) * (values[groups == g].mean() - grand_mean) ** 2 #sum of squares between groups
        for g in groups.unique()
    )
    ss_total = ((values - grand_mean) ** 2).sum() #total sum of squares
    return ss_between / ss_total if ss_total > 0 else 0

eta2_hour   = eta_squared(clean_tips['pickup_hour'].astype(str),     clean_tips['tip_rate']) #convert to string to treat as categorical
eta2_bucket = eta_squared(clean_tips['distance_bucket'].astype(str), clean_tips['tip_rate'])

print(f'---Association with tip_rate ---')
print(f'pickup_hour    : {eta2_hour}') #output full precision for comparison
print(f'distance_bucket : {eta2_bucket}')

winner = 'pickup_hour' if eta2_hour > eta2_bucket else 'distance_bucket' #to determine  the stronger association with tip_rate
print(f'\ntip_rate is more strongly associated with: {winner}')

---Association with tip_rate ---
pickup_hour    : 0.0025237876231725092
distance_bucket : 0.00319892628684413

tip_rate is more strongly associated with: distance_bucket


## D. Required Visualizations

**Import visualization libraries**

In [ ]:
#import visualization libraries
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# consistent style across all plots
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

### D1. Time Series of Daily Trips per Borough with 7-Day Rolling Average Overlaid

In [ ]:
# D1: Time series of daily trips for each borough with rolling 7-day average overlaid

daily_trip_counts['pickup_date'] = pd.to_datetime(daily_trip_counts['pickup_date'])

# exclude 'Unknown' borough for clarity
boroughs = [b for b in daily_trip_counts['PU_borough'].unique() if b != 'Unknown']

fig, axes = plt.subplots(len(boroughs), 1, figsize=(14, 3.5 * len(boroughs)), sharex=True)

for ax, borough in zip(axes, boroughs):
    subset = daily_trip_counts[daily_trip_counts['PU_borough'] == borough].sort_values('pickup_date')
    ax.plot(subset['pickup_date'], subset['trip_count'],
            alpha=0.55, linewidth=1.2, label='Daily trips', color='steelblue')
    ax.plot(subset['pickup_date'], subset['rolling_7d_mean'],
            linewidth=2.2, label='7-day rolling avg', color='darkorange')
    ax.set_title(borough, fontsize=11, fontweight='bold')
    ax.set_ylabel('Trip Count', fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.legend(fontsize=8, loc='upper left')

axes[-1].set_xlabel('Date (January 2023)', fontsize=10)
fig.suptitle('Daily Trip Counts per Borough — January 2023\n(with 7-Day Rolling Average)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../part1_taxi/plot1_daily_trips_borough.png', bbox_inches='tight')
plt.show()

### D2. Before/After Cleaning Distribution of `trip_duration_min`

In [ ]:
# D2: Before/after cleaning distribution of trip_duration_min

import pyarrow as pa
for ext_name in ['pandas.interval', 'pandas.period']:
    try:
        pa.unregister_extension_type(ext_name)
    except KeyError:
        pass

# load raw data and compute trip_duration_min before cleaning
df_raw = pd.read_parquet('../data/raw/tlc/yellow_tripdata_2023-01.parquet', engine='fastparquet')
df_raw['tpep_pickup_datetime']  = pd.to_datetime(df_raw['tpep_pickup_datetime'])
df_raw['tpep_dropoff_datetime'] = pd.to_datetime(df_raw['tpep_dropoff_datetime'])
df_raw['trip_duration_min'] = (
    (df_raw['tpep_dropoff_datetime'] - df_raw['tpep_pickup_datetime']).dt.total_seconds() / 60.0
)

# clip to a readable range for the histogram (0–120 min)
clip_max = 120
before_clipped = df_raw['trip_duration_min'].clip(lower=-10, upper=clip_max).dropna()
after_clipped  = df_clean['trip_duration_min'].clip(upper=clip_max)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

axes[0].hist(before_clipped, bins=100, color='salmon', edgecolor='none', alpha=0.85)
axes[0].set_title('Before Cleaning', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Trip Duration (minutes)', fontsize=10)
axes[0].set_ylabel('Number of Trips', fontsize=10)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].annotate(f'n = {len(df_raw):,}', xy=(0.97, 0.95), xycoords='axes fraction',
                  ha='right', fontsize=9, color='darkred')

axes[1].hist(after_clipped, bins=100, color='steelblue', edgecolor='none', alpha=0.85)
axes[1].set_title('After Cleaning', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Trip Duration (minutes)', fontsize=10)
axes[1].set_ylabel('Number of Trips', fontsize=10)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[1].annotate(f'n = {len(df_clean):,}', xy=(0.97, 0.95), xycoords='axes fraction',
                  ha='right', fontsize=9, color='darkblue')

fig.suptitle('Distribution of Trip Duration (minutes) — Before vs. After Cleaning\n'
             '(capped at 120 min for readability)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../part1_taxi/plot2_trip_duration_before_after.png', bbox_inches='tight')
plt.show()

### D3. Relationship Plot: `trip_distance` vs `fare_amount`

In [ ]:
# D3: Relationship plot: trip_distance vs fare_amount
# sample 50k points to keep the scatter readable

sample = df_clean[
    (df_clean['trip_distance'] <= 30) &
    (df_clean['fare_amount']   <= 100)
].sample(n=50_000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    sample['trip_distance'],
    sample['fare_amount'],
    c=sample['trip_distance'],
    cmap='plasma',
    alpha=0.25,
    s=6,
    linewidths=0
)

# add a linear trend line
m, b = np.polyfit(sample['trip_distance'], sample['fare_amount'], 1)
x_line = np.linspace(0, 30, 200)
ax.plot(x_line, m * x_line + b, color='red', linewidth=1.8, label=f'Trend  y = {m:.2f}x + {b:.2f}')

# correlation annotation
corr = sample['trip_distance'].corr(sample['fare_amount'])
ax.annotate(f'Pearson r = {corr:.3f}', xy=(0.03, 0.95), xycoords='axes fraction',
             fontsize=10, color='darkred')

cbar = fig.colorbar(scatter, ax=ax, pad=0.01)
cbar.set_label('Trip Distance (miles)', fontsize=9)

ax.set_title('Trip Distance vs. Fare Amount — NYC Yellow Taxi, January 2023\n'
             '(50,000 random trips; distance ≤ 30 mi, fare ≤ $100)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Trip Distance (miles)', fontsize=10)
ax.set_ylabel('Fare Amount (USD)', fontsize=10)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../part1_taxi/plot3_distance_vs_fare.png', bbox_inches='tight')
plt.show()

### D4. Heatmap of Average Trips by Weekday (rows) × Pickup Hour (columns)

In [ ]:
# D4: Heatmap of average trips by weekday (rows) and pickup_hour (columns)

# ordered weekday labels Mon -> Sun
weekday_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# mean trip count per (weekday, pickup_hour)
heatmap_data = (
    df_clean
    .groupby(['weekday', 'pickup_hour'])
    .size()
    .reset_index(name='trip_count')
)

heatmap_pivot = heatmap_data.pivot(index='weekday', columns='pickup_hour', values='trip_count')
heatmap_pivot = heatmap_pivot.reindex(weekday_order)  # order rows Mon-Sun

fig, ax = plt.subplots(figsize=(16, 5))

sns.heatmap(
    heatmap_pivot,
    ax=ax,
    cmap='YlOrRd',
    fmt=',d',
    linewidths=0.3,
    linecolor='white',
    annot=True,
    annot_kws={'size': 7},
    cbar_kws={'label': 'Total Trip Count'}
)

ax.set_title('Total Trip Count by Weekday × Pickup Hour — NYC Yellow Taxi, January 2023',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Pickup Hour (0 = midnight, 23 = 11 PM)', fontsize=10)
ax.set_ylabel('Weekday', fontsize=10)
ax.tick_params(axis='x', labelsize=8)
ax.tick_params(axis='y', labelsize=9, rotation=0)

plt.tight_layout()
plt.savefig('../part1_taxi/plot4_heatmap_weekday_hour.png', bbox_inches='tight')
plt.show()